# EduAdapt AI — Pipeline de Testes

Este notebook **reproduz fielmente** a pipeline da plataforma, passo a passo:

| Etapa | Célula | Arquivo da plataforma |
|---|---|---|
| 0. Setup | 1–2 | — |
| 1. Atividade + Perfil | 3–4 | `models/activity.py`, `models/student_profile.py` |
| 2. Estilos de imagem | 5 | `services/openai_service.py` → `IMAGE_STYLES` |
| 3. Adaptação de texto | 6–7 | `services/openai_service.py` → prompt principal |
| 4. Prompts de imagem | 8 | `services/openai_service.py` → `_make_prompts()` |
| 5. Geração real de imagens | 9–10 | `routes/adaptations.py` → `generate-images` |
| 6. Roteiro de áudio | 11–12 | `agents/audio_generator.py` |
| 7. Interação | 13–14 | `agents/interaction_generator.py` |
| 8. Validação | 15–16 | `agents/adaptation_validator.py` |
| 9. Pipeline completa (GPT) | 17–18 | `services/openai_service.py` → `generate_adaptation_with_ai()` |
| 10. Resultados | 19–20 | — |

## Como usar
1. Execute as células em ordem
2. Edite os prompts nas células marcadas com `✏️ EDITAR AQUI`
3. Reexecute a célula para ver o resultado
4. Para transferir melhorias para a plataforma: peça ao Claude qual arquivo atualizar (a tabela acima indica o caminho exato)

> **Custo estimado por execução completa:** ~$0.20–0.50 (8 imagens × $0.04 + texto GPT-4o-mini)

In [ ]:
# ─── INSTALAÇÃO (execute uma vez) ──────────────────────────────────────────
%pip install openai>=1.75.0 httpx nest_asyncio Pillow requests -q
print("✅ Dependências instaladas")

In [ ]:
# ─── CONFIGURAÇÃO ──────────────────────────────────────────────────────────
import os, asyncio, json, re, uuid
from datetime import datetime
from pathlib import Path
from typing import Optional
from IPython.display import display, Image as IPImage, Markdown
import nest_asyncio
nest_asyncio.apply()  # permite asyncio.run() dentro do Jupyter

# ✏️ Coloque sua chave aqui OU defina a variável de ambiente OPENAI_API_KEY
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

if not OPENAI_API_KEY or OPENAI_API_KEY.endswith("..."):
    print("⚠️  Defina OPENAI_API_KEY antes de continuar")
    print("   Exemplo: export OPENAI_API_KEY=sk-proj-...")
else:
    from openai import AsyncOpenAI
    client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    print(f"✅ Cliente OpenAI configurado  ({OPENAI_API_KEY[:12]}...)")

# Pasta para salvar imagens baixadas
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"📁 Resultados serão salvos em: {OUTPUT_DIR.absolute()}")

---
## 📋 Seção 1 — Atividade e Perfil do Aluno

Edite os dicionários abaixo para testar com diferentes atividades e perfis.

> **Plataforma:** `apps/api/app/models/activity.py` e `apps/api/app/models/student_profile.py`

In [ ]:
# ✏️ ATIVIDADE — mesmos campos do modelo Activity da plataforma
# Plataforma: apps/api/app/models/activity.py

ATIVIDADE = {
    "title":                "Animais do Brasil",
    "discipline":           "Ciências",
    "school_year":          "3º ano EF",
    "pedagogical_objective": "Identificar e classificar animais aquáticos e terrestres",
    "activity_type":        "drag_and_drop",   # drag_and_drop | sequencing | multiple_choice
    "statement":            "Observe os animais abaixo.",
    "question":             "Classifique cada animal em aquático ou terrestre.",
    "expected_answer":      "Peixe, Tubarão são aquáticos. Leão, Elefante são terrestres.",
    "correction_criteria": "Cada item classificado corretamente vale 1 ponto.",
    "teacher_notes":        "Aluno tem dificuldade com textos longos. Prefira imagens grandes e instruções curtas.",
}

print("✅ Atividade definida:", ATIVIDADE["title"])
print("   Tipo:", ATIVIDADE["activity_type"])
print("   Resposta esperada:", ATIVIDADE["expected_answer"])

In [ ]:
# ✏️ PERFIL DO ALUNO — mesmos campos do modelo StudentProfile da plataforma
# Plataforma: apps/api/app/models/student_profile.py

PERFIL = {
    "name":                   "Perfil TEA Nível 1",
    "description":            "Autismo nível 1, boa comunicação verbal, dificuldade de foco",
    "reading_level":          "basic",    # initial | basic | fluent
    "autonomy_level":         "medium",   # low | medium | high
    "main_difficulties":      ["foco prolongado", "leitura de textos longos", "transições abruptas"],
    "recommended_strategies": ["recursos visuais", "instruções curtas e objetivas", "reforço positivo"],
    "preferred_modalities":   ["visual", "imagem", "pictograma"],
    "resources_to_avoid":     ["texto longo", "muitas cores simultâneas", "sons altos"],
    "accessibility_complexity": "medium",
    "notes":                  "Responde bem a rotinas visuais.",
}

print("✅ Perfil definido:", PERFIL["name"])
print("   Leitura:", PERFIL["reading_level"], "| Autonomia:", PERFIL["autonomy_level"])
print("   Dificuldades:", ", ".join(PERFIL["main_difficulties"]))

---
## 🎨 Seção 2 — Estilos de Imagem

Define os estilos disponíveis para geração de imagens e os prompts de estilo.

> **Plataforma:** `apps/api/app/services/openai_service.py` → `IMAGE_STYLES`  
> **Para transferir:** substitua o dicionário `IMAGE_STYLES` no arquivo acima pelo dicionário editado aqui.

In [ ]:
# ✏️ ESTILOS DE IMAGEM — réplica EXATA de IMAGE_STYLES em openai_service.py
# Plataforma: apps/api/app/services/openai_service.py → IMAGE_STYLES
# ⚠️  Alterar aqui e pedir ao Claude para atualizar o arquivo da plataforma

IMAGE_STYLES = {
    "line_art": {
        "label":  "Desenho P&B — traços simples",
        "suffix": "simple black and white line drawing, educational, minimal clean lines, no color, sketch style, high contrast",
        "model":  "gpt-image-1",
        "size":   "1024x1024",
    },
    "cartoon_2d": {
        "label":  "Cartoon colorido — detalhes 2D",
        "suffix": "colorful 2D cartoon illustration, child-friendly, vibrant colors, simple shapes, cute small details, educational",
        "model":  "gpt-image-1",
        "size":   "1024x1024",
    },
}

def make_prompts(subject: str) -> dict:
    """Réplica de _make_prompts() em openai_service.py"""
    return {style: f"{subject}, {cfg['suffix']}" for style, cfg in IMAGE_STYLES.items()}

# Preview dos prompts para o assunto da atividade
subject = f"educational illustration of {ATIVIDADE['title']}"
for style, prompt in make_prompts(subject).items():
    print(f"\n[{style}]")
    print(f"  {prompt[:120]}...")

---
## 📝 Seção 3 — Adaptação de Texto

Adapta o enunciado e a pergunta da atividade para o perfil do aluno usando GPT-4o-mini.

> **Plataforma:** `apps/api/app/services/openai_service.py` → `generate_adaptation_with_ai()`  
> Seção relevante do prompt: início do bloco `text_adaptations`

In [ ]:
# ✏️ PROMPT — ADAPTAÇÃO DE TEXTO
# Plataforma: apps/api/app/services/openai_service.py → generate_adaptation_with_ai()
# Instrução relevante: "text_adaptations" no prompt principal
# ─────────────────────────────────────────────────────────────────

TEXT_ADAPTATION_PROMPT = """
Você é um especialista em educação especial e tecnologia assistiva.
Adapte o texto da atividade abaixo para o perfil pedagógico do aluno.

━━━━━━━━ ATIVIDADE ━━━━━━━━
Título: {title}
Disciplina: {discipline} — {school_year}
Objetivo pedagógico: {pedagogical_objective}
Tipo: {activity_type}
Enunciado: {statement}
Pergunta: {question}
Resposta esperada: {expected_answer}
Observações do professor: {teacher_notes}

━━━━━━━━ PERFIL DO ALUNO ━━━━━━━━
Nível de leitura: {reading_level}
Nível de autonomia: {autonomy_level}
Dificuldades principais: {main_difficulties}
Estratégias recomendadas: {recommended_strategies}
Recursos a evitar: {resources_to_avoid}

━━━━━━━━ REGRAS ━━━━━━━━
- Use frases curtas e diretas
- Evite vocabulário complexo
- Adapte ao nível de leitura: 'initial'=letras maiúsculas e palavras simples, 'basic'=frases curtas, 'fluent'=texto normal
- Se houver dificuldades com foco, divida em passos numerados
- Mantenha o objetivo pedagógico

Retorne SOMENTE JSON válido:
{{"content": "texto adaptado aqui", "changes": ["lista", "de", "mudanças", "feitas"]}}
"""

print("✅ Prompt de adaptação de texto definido")
print(f"   {len(TEXT_ADAPTATION_PROMPT.split())} palavras")

In [ ]:
# ─── EXECUTAR: Adaptação de Texto ────────────────────────────────────────────

async def adapt_text_ai(activity: dict, profile: dict) -> dict:
    """Usa GPT-4o-mini (igual à plataforma) para adaptar o texto."""
    def fmt(val):
        if isinstance(val, list): return ", ".join(str(v) for v in val) if val else "não informado"
        return str(val) if val else "não informado"

    prompt = TEXT_ADAPTATION_PROMPT.format(
        title=fmt(activity.get("title")),
        discipline=fmt(activity.get("discipline")),
        school_year=fmt(activity.get("school_year")),
        pedagogical_objective=fmt(activity.get("pedagogical_objective")),
        activity_type=fmt(activity.get("activity_type")),
        statement=fmt(activity.get("statement")),
        question=fmt(activity.get("question")),
        expected_answer=fmt(activity.get("expected_answer")),
        teacher_notes=fmt(activity.get("teacher_notes")),
        reading_level=fmt(profile.get("reading_level")),
        autonomy_level=fmt(profile.get("autonomy_level")),
        main_difficulties=fmt(profile.get("main_difficulties")),
        recommended_strategies=fmt(profile.get("recommended_strategies")),
        resources_to_avoid=fmt(profile.get("resources_to_avoid")),
    )

    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

result_text = asyncio.run(adapt_text_ai(ATIVIDADE, PERFIL))

print("─" * 60)
print("TEXTO ADAPTADO:")
print("─" * 60)
print(result_text["content"])
print("\nMUDANÇAS FEITAS:")
for change in result_text.get("changes", []):
    print(f"  • {change}")

---
## 🖼️ Seção 4 — Prompts de Imagem

Gera os prompts para cada slot de imagem (image_options + itens de interação).

> **Plataforma:** `apps/api/app/services/openai_service.py` → `_mock_adaptation()` / `_make_prompts()`  
> Os prompts são gerados na criação da adaptação e armazenados em `output_data`.

In [ ]:
# ─── GERAÇÃO DE PROMPTS DE IMAGEM ────────────────────────────────────────────
# Réplica de _mock_adaptation() para a parte de image_options
# Plataforma: apps/api/app/services/openai_service.py

def make_generated_slots() -> dict:
    """Réplica de _make_generated_slots() — slots vazios para cada estilo."""
    return {style: {"image_url": None, "generated_at": None} for style in IMAGE_STYLES}

def build_image_options(activity: dict) -> list:
    """Réplica da criação de image_options em _mock_adaptation()."""
    title = activity.get("title", "Atividade")
    subjects = [
        ("img_1", f"Ilustração de {title}",    f"educational illustration of {title}"),
        ("img_2", "Pictograma AAC",             f"AAC pictogram for {title}"),
    ]
    return [
        {
            "id": img_id,
            "description": description,
            "base_subject": subject,
            "prompts": make_prompts(subject),
            "generated": make_generated_slots(),
            "active_style": "cartoon_2d",
            "is_active": True,
            "image_url": None,
        }
        for img_id, description, subject in subjects
    ]

def build_interaction_items(expected_answer: str) -> list:
    """Réplica da criação de items em _mock_adaptation()."""
    raw = [w.strip() for w in re.split(r'[,;]+', re.sub(
        r'\s+(são|vivem|pertencem|fazem parte|estão)\s+[\w\s]+?(?=[,.\n]|$)', "",
        expected_answer, flags=re.IGNORECASE
    )) if 1 < len(w.strip()) < 40][:5]
    if not raw:
        raw = ["Opção A", "Opção B", "Opção C"]
    return [
        {
            "name": name,
            "image_prompt": f"simple educational illustration of {name}, cartoon style",
            "prompts": make_prompts(name),
            "generated": make_generated_slots(),
            "active_style": "cartoon_2d",
            "image_url": None,
        }
        for name in raw
    ]

IMAGE_OPTIONS = build_image_options(ATIVIDADE)
INTERACTION_ITEMS = build_interaction_items(ATIVIDADE["expected_answer"])

print(f"\n📌 image_options ({len(IMAGE_OPTIONS)} imagens):")
for img in IMAGE_OPTIONS:
    print(f"  [{img['id']}] {img['description']}")
    for style, prompt in img["prompts"].items():
        print(f"      {style}: {prompt[:80]}...")

print(f"\n📌 interaction items ({len(INTERACTION_ITEMS)} itens):")
for item in INTERACTION_ITEMS:
    print(f"  [{item['name']}]")
    for style, prompt in item["prompts"].items():
        print(f"      {style}: {prompt[:80]}...")

---
## 🤖 Seção 5 — Geração Real de Imagens (gpt-image-1)

Chama a API OpenAI para gerar as imagens, exatamente como o endpoint `/generate-images` da plataforma faz.

> **Plataforma:** `apps/api/app/routes/adaptations.py` → `generate_images()`  
> **Custo:** ~$0.04 por imagem com gpt-image-1  
> ⚠️ A chave `sk-proj-...` deve ter `gpt-image-1` habilitado no projeto OpenAI

In [ ]:
# ✏️ CONFIGURAÇÃO DE GERAÇÃO DE IMAGEM
# Plataforma: apps/api/app/routes/adaptations.py → generate_images()
# ─────────────────────────────────────────────────────────────────

STYLE_TO_GENERATE = "cartoon_2d"  # ← trocar para "line_art" para testar o outro estilo

async def generate_images_for_style(
    image_options: list,
    interaction_items: list,
    style: str,
) -> tuple[list, list, dict]:
    """
    Réplica EXATA do endpoint POST /adaptations/{id}/generate-images.
    Plataforma: apps/api/app/routes/adaptations.py
    """
    cfg = IMAGE_STYLES[style]
    model = cfg["model"]
    size  = cfg["size"]
    
    generated_count = 0
    errors = []

    # --- image_options ---
    updated_images = []
    for img in image_options:
        img = dict(img)  # cópia
        already = (img.get("generated") or {}).get(style, {}).get("image_url")
        if already:
            print(f"  ⏭️  {img['id']} já gerado — pulando")
            updated_images.append(img)
            continue
        prompt = (img.get("prompts") or {}).get(style) or ""
        if not prompt:
            updated_images.append(img)
            continue
        try:
            print(f"  🎨 Gerando {img['id']} [{style}]... ", end="", flush=True)
            resp = await client.images.generate(
                model=model, prompt=prompt[:1000], size=size, n=1, response_format="url"
            )
            url = resp.data[0].url
            if "generated" not in img:
                img["generated"] = {}
            img["generated"][style] = {"image_url": url, "generated_at": datetime.utcnow().isoformat()}
            if img.get("active_style", "cartoon_2d") == style:
                img["image_url"] = url
            generated_count += 1
            print("✅")
        except Exception as e:
            print("❌")
            errors.append({"id": img["id"], "error": str(e)})
        updated_images.append(img)

    # --- interaction items ---
    updated_items = []
    for item in interaction_items:
        item = dict(item)  # cópia
        already = (item.get("generated") or {}).get(style, {}).get("image_url")
        if already:
            print(f"  ⏭️  item '{item['name']}' já gerado — pulando")
            updated_items.append(item)
            continue
        prompt = (item.get("prompts") or {}).get(style) or item.get("image_prompt", "")
        if not prompt:
            updated_items.append(item)
            continue
        try:
            print(f"  🎨 Gerando item '{item['name']}' [{style}]... ", end="", flush=True)
            resp = await client.images.generate(
                model=model, prompt=prompt[:1000], size=size, n=1, response_format="url"
            )
            url = resp.data[0].url
            if "generated" not in item:
                item["generated"] = {}
            item["generated"][style] = {"image_url": url, "generated_at": datetime.utcnow().isoformat()}
            if item.get("active_style", "cartoon_2d") == style:
                item["image_url"] = url
            generated_count += 1
            print("✅")
        except Exception as e:
            print("❌")
            errors.append({"id": item["name"], "error": str(e)})
        updated_items.append(item)

    stats = {"images_generated": generated_count, "errors": errors}
    return updated_images, updated_items, stats

print("✅ Função de geração definida")
print(f"   Estilo selecionado: {STYLE_TO_GENERATE} ({IMAGE_STYLES[STYLE_TO_GENERATE]['model']})")

In [ ]:
# ─── EXECUTAR: Geração de Imagens ────────────────────────────────────────────
import requests
from PIL import Image as PILImage
from io import BytesIO

print(f"Gerando imagens no estilo '{STYLE_TO_GENERATE}'...\n")
IMAGE_OPTIONS_GENERATED, INTERACTION_ITEMS_GENERATED, gen_stats = asyncio.run(
    generate_images_for_style(IMAGE_OPTIONS, INTERACTION_ITEMS, STYLE_TO_GENERATE)
)

print(f"\n{'─'*50}")
print(f"✅ {gen_stats['images_generated']} imagens geradas")
if gen_stats["errors"]:
    print(f"❌ {len(gen_stats['errors'])} erros:")
    for err in gen_stats["errors"]:
        print(f"   {err['id']}: {err['error'][:120]}")

# Exibir imagens geradas inline
all_slots = [
    (img["id"], img["description"], img)
    for img in IMAGE_OPTIONS_GENERATED
] + [
    (item["name"], item["name"], item)
    for item in INTERACTION_ITEMS_GENERATED
]

for slot_id, label, slot in all_slots:
    url = (slot.get("generated") or {}).get(STYLE_TO_GENERATE, {}).get("image_url")
    if url:
        try:
            resp = requests.get(url, timeout=15)
            img = PILImage.open(BytesIO(resp.content))
            img_resized = img.resize((256, 256))
            # Salvar localmente
            fname = OUTPUT_DIR / f"{slot_id}_{STYLE_TO_GENERATE}.png"
            img_resized.save(fname)
            print(f"\n📸 {label}")
            display(IPImage(filename=str(fname), width=256))
        except Exception as e:
            print(f"   ⚠️ Não foi possível exibir {slot_id}: {e}")

---
## 🔊 Seção 6 — Roteiro de Áudio

Gera o roteiro de narração (TTS) adaptado ao nível de autonomia do aluno.

> **Plataforma:** `apps/api/app/agents/audio_generator.py`  
> No pipeline com IA, o roteiro é gerado dentro do prompt principal em `audio_options`.

In [ ]:
# ✏️ PROMPT — ROTEIRO DE ÁUDIO
# Plataforma: apps/api/app/agents/audio_generator.py + openai_service.py
# Para adaptar: edite o prompt abaixo e peça ao Claude para atualizar o arquivo
# ─────────────────────────────────────────────────────────────────

AUDIO_VOICE_STYLES = {
    "low":    "muito calmo e pausado, com pausas longas entre as instruções. Repita as instruções.",
    "medium": "claro e objetivo, com ritmo moderado",
    "high":   "direto ao ponto, ritmo normal",
}

AUDIO_SCRIPT_PROMPT = """
Você é um especialista em criação de roteiros para texto-fala (TTS) para alunos com necessidades especiais.
Crie um roteiro de narração para a atividade abaixo.

ATIVIDADE:
- Enunciado: {statement}
- Pergunta: {question}

ESTILO DE VOZ: {voice_style}

REGRAS:
- Use frases curtas (máximo 10 palavras por frase)
- Inclua pausas indicadas por "[PAUSA]"
- Comece com uma chamada de atenção
- Termine com instrução de ação clara
- Adapte o vocabulário ao nível de leitura: {reading_level}

Retorne SOMENTE o roteiro, sem JSON. Exemplo de formato:

Atenção. [PAUSA]
Vamos começar a atividade. [PAUSA]
Observe os animais. [PAUSA]
...
"""

print("✅ Prompt de áudio definido")
print("\nEstilos de voz disponíveis:")
for level, style in AUDIO_VOICE_STYLES.items():
    marker = "← ATUAL" if level == PERFIL["autonomy_level"] else ""
    print(f"  {level}: {style} {marker}")

In [ ]:
# ─── EXECUTAR: Roteiro de Áudio ────────────────────────────────────────────

async def generate_audio_ai(activity: dict, profile: dict) -> dict:
    """Gera roteiro de áudio adaptado. Réplica com IA do audio_generator.py"""
    autonomy = profile.get("autonomy_level", "medium")
    voice_style = AUDIO_VOICE_STYLES.get(autonomy, "claro e objetivo")

    prompt = AUDIO_SCRIPT_PROMPT.format(
        statement=activity.get("statement", ""),
        question=activity.get("question", ""),
        voice_style=voice_style,
        reading_level=profile.get("reading_level", "basic"),
    )

    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    script = resp.choices[0].message.content.strip()
    return {"id": "audio_1", "script": script, "voice_style": voice_style, "source": "ai"}

result_audio = asyncio.run(generate_audio_ai(ATIVIDADE, PERFIL))

print("─" * 60)
print("ROTEIRO DE ÁUDIO GERADO:")
print(f"Estilo de voz: {result_audio['voice_style']}")
print("─" * 60)
print(result_audio["script"])

---
## 🎮 Seção 7 — Estrutura de Interação

Deriva os itens e zonas da atividade interativa (drag_and_drop, sequencing, multiple_choice).

> **Plataforma:** `apps/api/app/agents/interaction_generator.py`  
> Os itens gerados aqui também recebem prompts de imagem (Seção 4).

In [ ]:
# ✏️ PROMPT — INTERAÇÃO (versão IA)
# A versão mock usa regex. Esta versão usa GPT para melhor interpretação.
# Plataforma: apps/api/app/agents/interaction_generator.py + openai_service.py (campo interaction_options)
# ─────────────────────────────────────────────────────────────────

INTERACTION_PROMPT = """
Você é um especialista em criação de atividades interativas para educação especial.
Analise a atividade abaixo e gere a estrutura de interação adequada.

ATIVIDADE:
- Tipo: {activity_type}
- Pergunta: {question}
- Resposta esperada: {expected_answer}

REGRAS POR TIPO:
- drag_and_drop: extraia itens (substantivos) e zonas (categorias). Mapeie cada item à sua zona.
- sequencing: extraia os passos da sequência. Numere as zonas (1°, 2°, 3°...). 
- multiple_choice: crie "Minha resposta" como único item e as opções como zonas.

Retorne SOMENTE JSON válido:
{{
  "type": "drag_and_drop",
  "instructions": "instrução direta ao aluno (frase curta)",
  "items": ["item1", "item2", "item3"],
  "zones": ["Zona A", "Zona B"],
  "correct_answer": {{"item1": "Zona A", "item2": "Zona B"}},
  "feedback_correct": "Muito bem!",
  "feedback_incorrect": "Tente novamente."
}}
"""

print("✅ Prompt de interação definido")
print(f"   Tipo da atividade: {ATIVIDADE['activity_type']}")

In [ ]:
# ─── EXECUTAR: Geração de Interação ────────────────────────────────────────

async def generate_interaction_ai(activity: dict) -> dict:
    """Versão IA do interaction_generator.py"""
    prompt = INTERACTION_PROMPT.format(
        activity_type=activity.get("activity_type", "drag_and_drop"),
        question=activity.get("question", ""),
        expected_answer=activity.get("expected_answer", ""),
    )
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    data = json.loads(resp.choices[0].message.content)
    # Adicionar prompts de imagem aos itens (igual à plataforma)
    items_with_prompts = []
    for item_name in data.get("items", []):
        items_with_prompts.append({
            "name": item_name,
            "image_prompt": f"simple educational illustration of {item_name}, cartoon style",
            "prompts": make_prompts(item_name),
            "generated": {style: {"image_url": None} for style in IMAGE_STYLES},
            "active_style": "cartoon_2d",
            "image_url": None,
        })
    data["items"] = items_with_prompts
    data["zones"] = [{"name": z} if isinstance(z, str) else z for z in data.get("zones", [])]
    return data

result_interaction = asyncio.run(generate_interaction_ai(ATIVIDADE))

print("─" * 60)
print("ESTRUTURA DE INTERAÇÃO GERADA:")
print("─" * 60)
print(f"Tipo: {result_interaction['type']}")
print(f"Instrução: {result_interaction['instructions']}")
print(f"\nItens ({len(result_interaction['items'])}):") 
for item in result_interaction["items"]:
    print(f"  • {item['name']}")
print(f"\nZonas ({len(result_interaction['zones'])}):") 
for zone in result_interaction["zones"]:
    print(f"  • {zone['name'] if isinstance(zone, dict) else zone}")
print(f"\nResposta correta:")
for item_name, zone in result_interaction.get("correct_answer", {}).items():
    print(f"  {item_name} → {zone}")

---
## ✅ Seção 8 — Validação da Adaptação

Avalia a qualidade pedagógica e de acessibilidade da adaptação gerada.

> **Plataforma:** `apps/api/app/agents/adaptation_validator.py`

In [ ]:
# ✏️ PROMPT — VALIDAÇÃO
# Plataforma: apps/api/app/agents/adaptation_validator.py
# O mock usa heurísticas simples. Esta versão usa GPT para avaliação real.
# ─────────────────────────────────────────────────────────────────

VALIDATION_PROMPT = """
Você é um especialista em educação especial. Avalie a adaptação pedagógica abaixo.

PERFIL DO ALUNO:
- Dificuldades: {main_difficulties}
- Estratégias recomendadas: {recommended_strategies}
- Nível de leitura: {reading_level}

ADAPTAÇÃO GERADA:
- Texto adaptado: {adapted_text}
- Número de imagens: {num_images}
- Roteiro de áudio: {has_audio}
- Tipo de interação: {interaction_type}
- Instrução de interação: {interaction_instruction}

Avalie em uma escala de 1-5 cada dimensão. Retorne SOMENTE JSON:
{{
  "clarity_score": 4,
  "accessibility_score": 5,
  "pedagogical_score": 4,
  "difficulty_score": 3,
  "approved": true,
  "notes": "justificativa resumida",
  "improvement_suggestions": ["sugestão 1", "sugestão 2"]
}}
"""

print("✅ Prompt de validação definido")

In [ ]:
# ─── EXECUTAR: Validação ────────────────────────────────────────────────────

async def validate_adaptation_ai(
    text_result: dict,
    image_options: list,
    audio_result: dict,
    interaction: dict,
    profile: dict,
) -> dict:
    """Versão IA do adaptation_validator.py"""
    def fmt(val):
        if isinstance(val, list): return ", ".join(str(v) for v in val)
        return str(val) if val else "não informado"

    prompt = VALIDATION_PROMPT.format(
        main_difficulties=fmt(profile.get("main_difficulties")),
        recommended_strategies=fmt(profile.get("recommended_strategies")),
        reading_level=fmt(profile.get("reading_level")),
        adapted_text=text_result.get("content", "")[:300],
        num_images=len(image_options),
        has_audio="sim" if audio_result else "não",
        interaction_type=interaction.get("type", ""),
        interaction_instruction=interaction.get("instructions", ""),
    )
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

result_validation = asyncio.run(validate_adaptation_ai(
    result_text, IMAGE_OPTIONS, result_audio, result_interaction, PERFIL
))

print("─" * 60)
print("VALIDAÇÃO:")
print("─" * 60)
scores = ["clarity_score", "accessibility_score", "pedagogical_score", "difficulty_score"]
for score in scores:
    val = result_validation.get(score, "?")
    bar = "█" * int(val) + "░" * (5 - int(val)) if isinstance(val, (int, float)) else ""
    print(f"  {score:<25} {bar} {val}/5")
print(f"\n  Aprovada: {'✅ SIM' if result_validation.get('approved') else '❌ NÃO'}")
print(f"  Notas: {result_validation.get('notes', '')}")
sugestoes = result_validation.get("improvement_suggestions", [])
if sugestoes:
    print("\n  Sugestões de melhoria:")
    for s in sugestoes:
        print(f"    • {s}")

---
## 🚀 Seção 9 — Pipeline Completa (GPT-4o-mini, igual à plataforma)

Esta é a chamada real que a plataforma faz: um único prompt JSON que gera toda a estrutura de uma vez.

> **Plataforma:** `apps/api/app/services/openai_service.py` → `generate_adaptation_with_ai()`  
> **Para melhorar:** edite `FULL_PIPELINE_PROMPT` abaixo e peça ao Claude para transferir para `openai_service.py`

In [ ]:
# ✏️ PROMPT PRINCIPAL DA PIPELINE COMPLETA
# Este prompt é IDÊNTICO ao usado em generate_adaptation_with_ai() na plataforma.
# Plataforma: apps/api/app/services/openai_service.py → generate_adaptation_with_ai()
# ─────────────────────────────────────────────────────────────────────────────────
# COMO MELHORAR: edite as seções abaixo, teste, e quando estiver satisfeito,
# peça ao Claude: "transfira o FULL_PIPELINE_PROMPT do notebook para a plataforma"
# ─────────────────────────────────────────────────────────────────────────────────

# Gera documentação de estilos dinamicamente (igual à plataforma)
STYLES_DOC = "\n".join(
    f'   - "{k}": prompts no estilo {v["label"]}' for k, v in IMAGE_STYLES.items()
)

def build_full_pipeline_prompt(activity: dict, profile: dict) -> str:
    """Constrói o prompt exato de generate_adaptation_with_ai()."""
    def fmt(val):
        if val is None or val == "": return "não informado"
        if isinstance(val, list): return ", ".join(str(v) for v in val) if val else "não informado"
        return str(val)

    # Detecta tipo de interação (replica _detect_interaction_type)
    text = " ".join([
        fmt(activity.get("question")),
        fmt(activity.get("activity_type")),
        fmt(activity.get("pedagogical_objective")),
    ]).lower()
    if any(k in text for k in ["ordem", "ordenar", "sequência", "sequencia", "rotina", "passo"]):
        detected_type = "sequencing"
    elif any(k in text for k in ["qual é", "escolha", "marque", "assinale"]):
        detected_type = "multiple_choice"
    else:
        detected_type = "drag_and_drop"

    return f"""Você é um especialista em educação especial e tecnologia assistiva.
Adapte a atividade abaixo para o perfil pedagógico do aluno descrito.

━━━━━━━━ ATIVIDADE ━━━━━━━━
Título: {fmt(activity.get("title"))}
Disciplina: {fmt(activity.get("discipline"))}
Ano escolar: {fmt(activity.get("school_year"))}
Objetivo pedagógico: {fmt(activity.get("pedagogical_objective"))}
Tipo: {fmt(activity.get("activity_type"))} (detectado: {detected_type})
Enunciado: {fmt(activity.get("statement"))}
Pergunta: {fmt(activity.get("question"))}
Resposta esperada: {fmt(activity.get("expected_answer"))}
Observações do professor: {fmt(activity.get("teacher_notes"))}

━━━━━━━━ PERFIL DO ALUNO ━━━━━━━━
Nome do perfil: {fmt(profile.get("name"))}
Nível de leitura: {fmt(profile.get("reading_level"))}
Nível de autonomia: {fmt(profile.get("autonomy_level"))}
Dificuldades principais: {fmt(profile.get("main_difficulties"))}
Estratégias recomendadas: {fmt(profile.get("recommended_strategies"))}
Modalidades preferidas: {fmt(profile.get("preferred_modalities"))}
Recursos a evitar: {fmt(profile.get("resources_to_avoid"))}

━━━━━━━━ INSTRUÇÕES CRÍTICAS ━━━━━━━━
1. interaction_options[0].type: "{detected_type}" (baseado na atividade)
   - "sequencing": ordenar/sequenciar — zones=[{{"name":"1°"}}, {{"name":"2°"}},...]
   - "multiple_choice": resposta única — items=[{{"name":"Minha resposta"}}], zones=opções
   - "drag_and_drop": classificar em categorias

2. items[]: cada objeto deve ter:
   - "name": texto do item
   - "image_prompt": prompt curto em inglês
   - "prompts": {{"line_art": "...", "cartoon_2d": "..."}} em inglês
   - "generated": {{"line_art": {{"image_url": null}}, "cartoon_2d": {{"image_url": null}}}}
   - "active_style": "cartoon_2d"
   - "image_url": null

3. zones[]: objetos {{"name": "rótulo"}}

4. correct_answer: {{"nome_item": "nome_zona"}} ou {{"correct_zone": "zona"}} para multiple_choice

5. image_options[]: 2 itens com estilos:
{STYLES_DOC}
   Cada item: {{"id", "description" (pt-BR), "base_subject" (inglês), "prompts", "generated", "active_style":"cartoon_2d", "is_active":true, "image_url":null}}

6. audio_options[]: roteiro TTS curto e pausado, estilo de voz adaptado ao nível de autonomia

7. Todos os textos em português brasileiro. Prompts de imagem em inglês.

Responda APENAS com JSON válido no formato:
{{
  "text_adaptations": [{{"version": 1, "content": "texto adaptado"}}],
  "image_options": [...],
  "audio_options": [{{"id": "audio_1", "script": "...", "voice_style": "..."}}],
  "interaction_options": [{{...}}],
  "print_version": {{"format": "A4", "layout": "single_column", "font_size": "large", "instructions": "...", "answer_space": true}},
  "validation": {{"clarity_score": 5, "accessibility_score": 5, "pedagogical_score": 5, "difficulty_score": 3, "approved": true, "notes": "..."}}
}}"""

print("✅ Prompt da pipeline completa definido")
print(f"   {len(build_full_pipeline_prompt(ATIVIDADE, PERFIL).split())} palavras")

In [ ]:
# ─── EXECUTAR: Pipeline Completa ─────────────────────────────────────────────

async def run_full_pipeline(activity: dict, profile: dict) -> dict:
    """Réplica EXATA de generate_adaptation_with_ai() em openai_service.py"""
    prompt = build_full_pipeline_prompt(activity, profile)
    resp = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

print("Executando pipeline completa (GPT-4o-mini)...")
result_full_pipeline = asyncio.run(run_full_pipeline(ATIVIDADE, PERFIL))
print("✅ Pipeline concluída!")

# Preview dos resultados
print("\n" + "─" * 60)
print("📄 TEXTO ADAPTADO:")
print(result_full_pipeline.get("text_adaptations", [{}])[0].get("content", ""))

print("\n📌 IMAGE OPTIONS:")
for img in result_full_pipeline.get("image_options", []):
    print(f"  [{img['id']}] {img.get('description', '')}")
    for style, prompt in (img.get("prompts") or {}).items():
        print(f"      {style}: {str(prompt)[:80]}...")

print("\n🔊 ÁUDIO:")
audio = result_full_pipeline.get("audio_options", [{}])[0]
print(f"  Estilo: {audio.get('voice_style', '')}")
print(f"  {audio.get('script', '')[:200]}...")

interaction = (result_full_pipeline.get("interaction_options") or [{}])[0]
print("\n🎮 INTERAÇÃO:")
print(f"  Tipo: {interaction.get('type', '')}")
print(f"  Instrução: {interaction.get('instructions', '')}")
print(f"  Itens: {[i['name'] if isinstance(i, dict) else i for i in interaction.get('items', [])]}")
print(f"  Zonas: {[z['name'] if isinstance(z, dict) else z for z in interaction.get('zones', [])]}")
print(f"  Resposta correta: {interaction.get('correct_answer', {})}")

validation = result_full_pipeline.get("validation", {})
print("\n✅ VALIDAÇÃO:")
for key in ["clarity_score", "accessibility_score", "pedagogical_score", "difficulty_score"]:
    print(f"  {key}: {validation.get(key, '?')}/5")
print(f"  Aprovada: {'✅' if validation.get('approved') else '❌'} — {validation.get('notes', '')}")

---
## 💾 Seção 10 — Salvar e Revisar Resultados

Salva a saída completa da pipeline em JSON para revisão posterior.

In [ ]:
# ─── SALVAR RESULTADOS ────────────────────────────────────────────────────────

full_result = {
    "metadata": {
        "timestamp": datetime.utcnow().isoformat(),
        "activity_title": ATIVIDADE["title"],
        "profile_name": PERFIL["name"],
        "style_generated": STYLE_TO_GENERATE,
    },
    # Resultado da pipeline completa (GPT-4o-mini)
    "full_pipeline": result_full_pipeline,
    # Resultados dos agentes individuais (para comparação)
    "individual_agents": {
        "text": result_text,
        "audio": result_audio,
        "interaction": {
            **result_interaction,
            "items": [
                {**item, "prompts": {k: v[:60]+"..." for k,v in item.get("prompts", {}).items()}}
                for item in result_interaction.get("items", [])
            ]
        },
        "validation": result_validation,
    },
    # Imagens geradas
    "images": {
        "style": STYLE_TO_GENERATE,
        "image_options": [
            {
                "id": img["id"],
                "description": img["description"],
                "url": (img.get("generated") or {}).get(STYLE_TO_GENERATE, {}).get("image_url"),
            }
            for img in IMAGE_OPTIONS_GENERATED
        ],
        "interaction_items": [
            {
                "name": item["name"],
                "url": (item.get("generated") or {}).get(STYLE_TO_GENERATE, {}).get("image_url"),
            }
            for item in INTERACTION_ITEMS_GENERATED
        ],
    },
}

output_file = OUTPUT_DIR / f"result_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(full_result, f, ensure_ascii=False, indent=2)

print(f"✅ Resultados salvos em: {output_file}")
print(f"   Tamanho: {output_file.stat().st_size / 1024:.1f} KB")

---
## 🗺️ Como transferir melhorias para a plataforma

Quando você ajustar um prompt neste notebook e quiser aplicar na plataforma, use este mapa:

| O que você editou no notebook | Arquivo da plataforma | O que pedir ao Claude |
|---|---|---|
| `TEXT_ADAPTATION_PROMPT` | `apps/api/app/services/openai_service.py` | "Substitua a seção de text_adaptations no prompt de generate_adaptation_with_ai" |
| `IMAGE_STYLES` | `apps/api/app/services/openai_service.py` | "Atualize o dicionário IMAGE_STYLES" |
| `AUDIO_SCRIPT_PROMPT` | `apps/api/app/agents/audio_generator.py` | "Atualize o prompt de geração de áudio" |
| `INTERACTION_PROMPT` | `apps/api/app/agents/interaction_generator.py` | "Substitua a lógica de geração de interação por esta versão IA" |
| `VALIDATION_PROMPT` | `apps/api/app/agents/adaptation_validator.py` | "Substitua a validação mock por esta versão com GPT" |
| `FULL_PIPELINE_PROMPT` (via `build_full_pipeline_prompt`) | `apps/api/app/services/openai_service.py` → `generate_adaptation_with_ai()` | "Transfira o FULL_PIPELINE_PROMPT do notebook para a plataforma" |
| `STYLE_TO_GENERATE` | `apps/web/app/(admin)/adaptations/[id]/review/page.tsx` | "Mude o estilo padrão de geração" |

### Exemplo de pedido eficiente ao Claude:

```
No notebook de testes, melhorei o AUDIO_SCRIPT_PROMPT para incluir marcadores de
pausas e adaptar ao nível de leitura. O novo prompt está na célula 'audio-prompt'.
Transfira essa melhoria para a plataforma.
```

Claude vai identificar o arquivo correto e fazer a substituição preservando o restante do código.

In [ ]:
# ─── COMPARAÇÃO: Mock vs IA ──────────────────────────────────────────────────
# Útil para avaliar quanto a IA melhora em relação ao mock

import sys
sys.path.insert(0, "../apps/api")  # ajuste se necessário

try:
    from app.services.openai_service import _mock_adaptation
    result_mock = _mock_adaptation(
        {
            "title": ATIVIDADE["title"],
            "statement": ATIVIDADE["statement"],
            "question": ATIVIDADE["question"],
            "expected_answer": ATIVIDADE["expected_answer"],
            "activity_type": ATIVIDADE["activity_type"],
        },
        {
            "preferred_modalities": PERFIL["preferred_modalities"],
            "main_difficulties": PERFIL["main_difficulties"],
        },
    )

    print("\n📊 COMPARAÇÃO: Mock vs Pipeline IA")
    print("─" * 60)
    print("MOCK — Texto adaptado:")
    print(result_mock["text_adaptations"][0]["content"][:300])
    print("\nPIPELINE IA — Texto adaptado:")
    print(result_full_pipeline.get("text_adaptations", [{}])[0].get("content", "")[:300])
except ImportError:
    print("⚠️  Não foi possível importar o mock da plataforma.")
    print("   Execute este notebook a partir da pasta raiz do projeto.")
    print("   Ou ajuste sys.path acima para o caminho correto do apps/api.")